In [1]:
import pandas as pd
import geopandas as gpd
import rasterio
print("all libraries loaded!")

all libraries loaded!


In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import matplotlib.pyplot as plt

print("libraries loaded!")

libraries loaded!


In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import matplotlib.pyplot as plt

print("libraries loaded!")

libraries loaded!


In [4]:
import rasterio
import numpy as np

# Load the raster
raster_path = r'C:\Users\moanna\Downloads\project2_barisal\barisal_swf_2015_2025.tif'

with rasterio.open(raster_path) as src:
    print('CRS:', src.crs)
    print('Resolution:', src.res)
    print('Shape:', src.shape)
    print('Min/Max values:', src.read(1).min(), src.read(1).max())

CRS: EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Shape: (14570, 12700)
Min/Max values: nan nan


In [5]:
with rasterio.open(raster_path) as src:
    data = src.read(1, masked=True)
    print('Min:', data.min())
    print('Max:', data.max())
    print('No data value:', src.nodata)
    print('Valid pixels:', data.count())

Min: nan
Max: nan
No data value: None
Valid pixels: 185039000


In [6]:
with rasterio.open(raster_path) as src:
    data = src.read(1)
    print('Data type:', data.dtype)
    print('Unique values sample:', np.unique(data[:100, :100]))
    print('Any non-nan values?', np.any(~np.isnan(data)))

Data type: float32
Unique values sample: [nan]
Any non-nan values? True


In [7]:
with rasterio.open(raster_path) as src:
    data = src.read(1)
    valid = data[~np.isnan(data)]
    print('Number of valid pixels:', len(valid))
    print('Min:', valid.min())
    print('Max:', valid.max())
    print('Unique values:', np.unique(valid))

Number of valid pixels: 117000928
Min: 0.0
Max: 11.0
Unique values: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [8]:
import os

folder = r'C:\Users\moanna\Downloads\project2_barisal'
for f in os.listdir(folder):
    print(f)

barisal_frequency_map.png
barisal_swf_2015_2025.tif
barisal_swf_2015_2025_t14.tif
barisal_swf_2015_2025_t18.tif
bgd_admin_boundaries.shp
bgd_admin_boundaries.shp.zip


In [9]:
gdf = gpd.read_file(r'C:\Users\moanna\Downloads\project2_barisal\bgd_admin_boundaries.shp')
print(gdf.columns.tolist())
print(gdf.shape)
print(gdf.head())

['iso2', 'iso3', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'version', 'area_sqkm', 'lang', 'lang1', 'lang2', 'lang3', 'adm0_ref_n', 'center_lat', 'center_lon', 'geometry']
(1, 19)
  iso2 iso3   adm0_name adm0_name1 adm0_name2 adm0_name3 adm0_pcode  \
0   BD  BDG  Bangladesh       None       None       None         BD   

    valid_on valid_to version      area_sqkm lang lang1 lang2 lang3  \
0 2023-05-21      NaT     v03  145962.104388   en  None  None  None   

  adm0_ref_n  center_lat  center_lon  \
0       None   23.769252   90.285391   

                                            geometry  
0  MULTIPOLYGON (((88.41501 26.63199, 88.41529 26...  


C:\Users\moanna\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'bgd_admin_boundaries.shp': 'bgd_admin0' (default), 'bgd_admin1', 'bgd_admin2', 'bgd_admin3', 'bgd_admincapitals', 'bgd_adminlines', 'bgd_adminpoints'. Specify layer parameter to avoid this warning.
  result = read_func(


In [10]:
gdf = gpd.read_file(
    r'C:\Users\moanna\Downloads\project2_barisal\bgd_admin_boundaries.shp',
    layer='bgd_admin3'
)
print(gdf.shape)
print(gdf.columns.tolist())

(507, 32)
['adm3_name', 'adm3_name1', 'adm3_name2', 'adm3_name3', 'adm3_pcode', 'adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'adm3_ref_n', 'center_lat', 'center_lon', 'geometry']


In [11]:
barisal_upazilas = gdf[gdf['adm1_name'] == 'Barisal']
print('Number of upazilas in Barisal:', len(barisal_upazilas))
print(barisal_upazilas['adm3_name'].tolist())

Number of upazilas in Barisal: 0
[]


In [12]:
print(gdf['adm1_name'].unique())

<StringArray>
['Chattogram',      'Dhaka',     'Khulna',   'Rajshahi',   'Barishal',
 'Mymensingh',    'Rangpur',     'Sylhet']
Length: 8, dtype: str


In [13]:
print('Upazila CRS:', barisal_upazilas.crs)

Upazila CRS: EPSG:4326


In [14]:
stats = zonal_stats(
    barisal_upazilas,
    raster_path,
    stats=['mean', 'max', 'min', 'std'],
    nodata=float('nan')
)

# Add results back to the geodataframe
barisal_upazilas = barisal_upazilas.copy()
barisal_upazilas['mean_frequency'] = [s['mean'] for s in stats]
barisal_upazilas['max_frequency'] = [s['max'] for s in stats]
barisal_upazilas['min_frequency'] = [s['min'] for s in stats]
barisal_upazilas['std_frequency'] = [s['std'] for s in stats]

print(barisal_upazilas[['adm3_name', 'mean_frequency']].sort_values('mean_frequency', ascending=False))

ValueError: Object is not a recognized source of Features

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

# Create spatial weights matrix
barisal_upazilas_reset = barisal_upazilas.reset_index(drop=True)
w = Queen.from_dataframe(barisal_upazilas_reset)
w.transform = 'r'  # row standardise

# Run Moran's I
moran = Moran(barisal_upazilas_reset['mean_frequency'], w, permutations=999)

print("Moran's I statistic:", round(moran.I, 4))
print("p-value:", round(moran.p_sim, 4))
print("z-score:", round(moran.z_sim, 4))

# Moran's I result interpretation
# I = 0.51, p = 0.001, z = 5.89
# Strong positive spatial autocorrelation
# Surface water frequency clusters significantly across Barisal upazilas
# High frequency areas are geographically adjacent to other high frequency areas

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 12))

barisal_upazilas_reset.plot(
    column='mean_frequency',
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Mean Surface Water Frequency (out of 11)'},
    ax=ax,
    edgecolor='black',
    linewidth=0.5
)

ax.set_title('Monsoon Surface Water Signal Frequency\nBarisal Division, Bangladesh 2015–2025', 
             fontsize=14, fontweight='bold')
ax.set_axis_off()

plt.tight_layout()
plt.savefig(r'C:\Users\moanna\Downloads\project2_barisal\barisal_frequency_map.png', 
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from rasterstats import zonal_stats
import numpy as np

# Test three threshold values
thresholds = [-14, -16, -18]  # -16 is our baseline

results = {}

for t in thresholds:
    with rasterio.open(raster_path) as src:
        # We need to rebuild the frequency raster at each threshold
        # For sensitivity testing we'll compare single year (2020) masks
        data = src.read(1)
    
    # Apply threshold
    mask = (data < t).astype(float)
    mask[np.isnan(data)] = np.nan
    
    results[t] = mask

# Compare -14 vs -16 (baseline) using Jaccard similarity
baseline = results[-16]
alternative1 = results[-14]
alternative2 = results[-18]

def jaccard(a, b):
    valid = ~np.isnan(a) & ~np.isnan(b)
    a_bin = a[valid] > 0
    b_bin = b[valid] > 0
    intersection = np.sum(a_bin & b_bin)
    union = np.sum(a_bin | b_bin)
    return intersection / union

print('Jaccard similarity -16 vs -14:', round(jaccard(baseline, alternative1), 4))
print('Jaccard similarity -16 vs -18:', round(jaccard(baseline, alternative2), 4))

In [ ]:
def jaccard(a, b):
    valid = (~np.isnan(a)) & (~np.isnan(b))
    if valid.sum() == 0:
        return np.nan
    a_bin = a[valid] > 0
    b_bin = b[valid] > 0
    intersection = np.sum(a_bin & b_bin)
    union = np.sum(a_bin | b_bin)
    if union == 0:
        return np.nan
    return intersection / union

# Check valid pixels in baseline
valid_pixels = np.sum(~np.isnan(results[-16]))
print('Valid pixels in baseline:', valid_pixels)
print('Non-zero pixels:', np.sum(results[-16][~np.isnan(results[-16])] > 0))

print('Jaccard -16 vs -14:', round(jaccard(results[-16], results[-14]), 4))
print('Jaccard -16 vs -18:', round(jaccard(results[-16], results[-18]), 4))

In [ ]:
import rasterio
import numpy as np

# Load all three rasters
def load_valid(path):
    with rasterio.open(path) as src:
        data = src.read(1)
    return data

baseline = load_valid(r'C:\Users\moanna\Downloads\project2_barisal\barisal_swf_2015_2025.tif')
t14 = load_valid(r'C:\Users\moanna\Downloads\project2_barisal\barisal_swf_2015_2025_t14.tif')
t18 = load_valid(r'C:\Users\moanna\Downloads\project2_barisal\barisal_swf_2015_2025_t18.tif')

print('Baseline shape:', baseline.shape)
print('t14 shape:', t14.shape)
print('t18 shape:', t18.shape)

In [ ]:
def jaccard(a, b, threshold=0):
    valid = (~np.isnan(a)) & (~np.isnan(b))
    a_bin = a[valid] > threshold
    b_bin = b[valid] > threshold
    intersection = np.sum(a_bin & b_bin)
    union = np.sum(a_bin | b_bin)
    if union == 0:
        return np.nan
    return intersection / union

# Compare baseline vs t14 and t18
j_14 = jaccard(baseline, t14)
j_18 = jaccard(baseline, t18)

print('Jaccard similarity baseline vs t14:', round(j_14, 4))
print('Jaccard similarity baseline vs t18:', round(j_18, 4))
print()
print('Interpretation:')
print('1.0 = identical, 0.0 = completely different')

In [ ]:
# Relative area error between baseline and alternatives
def relative_area_error(a, b):
    valid = (~np.isnan(a)) & (~np.isnan(b))
    area_a = np.sum(a[valid] > 0)
    area_b = np.sum(b[valid] > 0)
    return abs(area_a - area_b) / area_a

rae_14 = relative_area_error(baseline, t14)
rae_18 = relative_area_error(baseline, t18)

print('Relative area error baseline vs t14:', round(rae_14, 4))
print('Relative area error baseline vs t18:', round(rae_18, 4))

In [ ]:
# Export upazila data with frequency values to shapefile
barisal_upazilas_reset.to_file(
    r'C:\Users\moanna\Downloads\project2_barisal\barisal_upazilas_frequency.shp'
)
print("exported!")

In [ ]:
# Export upazila data with frequency values to shapefile
barisal_upazilas_reset.to_file(
    r'C:\Users\moanna\Downloads\project2_barisal\barisal_upazilas_frequency.shp'
)
print("exported!")

In [ ]:
```
NameError                                 Traceback (most recent call last)
Cell In[4], line 2
      1 # Export upazila data with frequency values to shapefile
----> 2 barisal_upazilas_reset.to_file(
      3     r'C:\Users\moanna\Downloads\project2_barisal\barisal_upazilas_frequency.shp'
      4 )
      5 print("exported!")

NameError: name 'barisal_upazilas_reset' is not defined
```

In [15]:
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio

raster_path = r'C:\Users\moanna\Downloads\project2_barisal\barisal_swf_2015_2025.tif'

# Load upazila boundaries
gdf = gpd.read_file(
    r'C:\Users\moanna\Downloads\project2_barisal\bgd_admin_boundaries.shp',
    layer='bgd_admin3'
)

# Filter to Barishal
barisal_upazilas = gdf[gdf['adm1_name'] == 'Barishal']

# Run zonal stats
stats = zonal_stats(
    barisal_upazilas,
    raster_path,
    stats=['mean', 'max', 'min', 'std'],
    nodata=float('nan')
)

barisal_upazilas = barisal_upazilas.copy()
barisal_upazilas['mean_frequency'] = [s['mean'] for s in stats]
barisal_upazilas['max_frequency'] = [s['max'] for s in stats]
barisal_upazilas['min_frequency'] = [s['min'] for s in stats]
barisal_upazilas['std_frequency'] = [s['std'] for s in stats]

print(barisal_upazilas[['adm3_name', 'mean_frequency']].sort_values('mean_frequency', ascending=False))

                     adm3_name  mean_frequency
338                 Daulatkhan        6.392369
331                      Hijla        5.298975
341                 Tazumuddin        4.988132
332                Mehendiganj        4.166278
335                Bhola Sadar        4.053313
336                Borhanuddin        3.227345
346                    Bauphal        2.882986
355                  Indurkani        2.415566
348                      Dumki        2.311065
354                  Bhandaria        2.281654
356                   Kawkhali        2.272943
329   Barishal Sadar (Kotwali)        2.192376
347                   Dashmina        2.162041
357                  Mathbaria        2.140605
323                Patharghata        2.030293
345                    Rajapur        2.027727
327                  Bakerganj        1.912816
324                    Taltali        1.856809
321              Barguna Sadar        1.831497
351                  Mirzaganj        1.819553
340          

In [16]:
barisal_upazilas_reset = barisal_upazilas.reset_index(drop=True)
barisal_upazilas_reset.to_file(
    r'C:\Users\moanna\Downloads\project2_barisal\barisal_upazilas_frequency.shp'
)
print("exported!")

exported!


C:\Users\moanna\AppData\Local\Temp\ipykernel_31976\3092388972.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  barisal_upazilas_reset.to_file(
C:\Users\moanna\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field valid_on created as String field, though DateTime requested.
  ogr_write(
C:\Users\moanna\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field valid_to created as String field, though DateTime requested.
  ogr_write(
C:\Users\moanna\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'mean_frequency' to 'mean_frequ'
  ogr_write(
C:\Users\moanna\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'max_frequency' to 'max_freque'
  ogr_write(
C:\Users\moanna\AppData\Local\Python\pyth

In [17]:
import os
print(os.listdir(r'C:\Users\moanna\Downloads\project2_barisal'))

['barisal_frequency_map.png', 'barisal_swf_2015_2025.tif', 'barisal_swf_2015_2025_t14.tif', 'barisal_swf_2015_2025_t18.tif', 'barisal_upazilas_frequency.cpg', 'barisal_upazilas_frequency.dbf', 'barisal_upazilas_frequency.prj', 'barisal_upazilas_frequency.shp', 'barisal_upazilas_frequency.shx', 'bgd_admin_boundaries.shp', 'bgd_admin_boundaries.shp.zip']


## Spearman Rank Correlation — Threshold Sensitivity

Tests whether upazila rankings are stable under threshold variation (±2dB around baseline -16dB).

All three pairwise correlations are high (r > 0.89, p < 0.001), indicating that the spatial ranking 
of upazilas by surface water frequency is robust to threshold variation within the tested range. 
The eastern clustering pattern is not an artefact of the chosen threshold value.

The weakest correlation (t14 vs t18, r = 0.891) represents the widest threshold gap tested (4dB). 
Even at this extreme, rankings remain substantially consistent.

# Spearman Rank Correlation across threshold scenarios
# Tests whether upazila rankings are stable under threshold variation

from scipy import stats
import pandas as pd

# Load zonal statistics for each threshold scenario
data_dir = r"C:\Users\moanna\Downloads\project2_barisal"

# Read the three rasters and compute zonal stats for each
from rasterstats import zonal_stats
import geopandas as gpd

upazilas = gpd.read_file(f"{data_dir}/barisal_upazilas_frequency.shp")

# Zonal stats for each threshold
stats_t14 = zonal_stats(upazilas, f"{data_dir}/barisal_swf_2015_2025_t14.tif", stats=["mean"])
stats_t16 = zonal_stats(upazilas, f"{data_dir}/barisal_swf_2015_2025.tif", stats=["mean"])
stats_t18 = zonal_stats(upazilas, f"{data_dir}/barisal_swf_2015_2025_t18.tif", stats=["mean"])

# Build comparison dataframe
df = pd.DataFrame({
    'upazila': upazilas['ADM3_EN'] if 'ADM3_EN' in upazilas.columns else upazilas.index,
    'mean_t14': [s['mean'] for s in stats_t14],
    'mean_t16': [s['mean'] for s in stats_t16],
    'mean_t18': [s['mean'] for s in stats_t18]
})

# Spearman rank correlations
corr_t14_t16, p_t14_t16 = stats.spearmanr(df['mean_t14'], df['mean_t16'])
corr_t14_t18, p_t14_t18 = stats.spearmanr(df['mean_t14'], df['mean_t18'])
corr_t16_t18, p_t16_t18 = stats.spearmanr(df['mean_t16'], df['mean_t18'])

print("Spearman Rank Correlations across threshold scenarios:")
print(f"t14 vs t16: r = {corr_t14_t16:.3f}, p = {p_t14_t16:.4f}")
print(f"t14 vs t18: r = {corr_t14_t18:.3f}, p = {p_t14_t18:.4f}")
print(f"t16 vs t18: r = {corr_t16_t18:.3f}, p = {p_t16_t18:.4f}")

print("\nInterpretation:")
print("r > 0.9 = rankings highly stable across thresholds")
print("r 0.7-0.9 = moderate stability")
print("r < 0.7 = threshold choice substantially affects rankings")